# Playground for Marint Naturkart hard/soft bottom

- Loads NGU sediment data and supporting region layers (municipalities and sea region)
- Classifies sediment polygons into BunnType categories: løsbunn, fastbunn, or uspesifisert
- Clips data to the sea area within selected municipalities
- Identifies areas missing bottom classification by differencing sea coverage and sediment union
- Appends missing areas as placeholder rows with BunnType="missing"
- Some of the datasets can be explored on the test terriamap [here](https://terriamap.t.niva.no/#start=%7B%22version%22%3A%228.0.0%22%2C%22initSources%22%3A%5B%7B%22stratum%22%3A%22user%22%2C%22models%22%3A%7B%22%2F%2FNGU+-+MarinBunnsedimenterWMS%22%3A%7B%22isOpen%22%3Atrue%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%2C%22ngu_marinbunnsedimenter%22%3A%7B%22isOpen%22%3Atrue%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%2FNGU+-+MarinBunnsedimenterWMS%22%5D%2C%22type%22%3A%22wms-group%22%7D%2C%22%2F%2FMarint+Natur+Kart%2FBunntyper+Union+AOI+%22%3A%7B%22show%22%3Atrue%2C%22opacity%22%3A1%2C%22activeStyle%22%3A%22BunnType%22%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%2FMarint+Natur+Kart%22%5D%2C%22type%22%3A%22geojson%22%7D%2C%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%2FSedimentkornstorrelse%2FKornstorrelse_Det%22%3A%7B%22show%22%3Atrue%2C%22isOpenInWorkbench%22%3Afalse%2C%22opacity%22%3A0.35%2C%22knownContainerUniqueIds%22%3A%5B%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%2FSedimentkornstorrelse%22%5D%2C%22type%22%3A%22wms%22%7D%2C%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%2FSedimentkornstorrelse%22%3A%7B%22isOpen%22%3Atrue%2C%22knownContainerUniqueIds%22%3A%5B%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%22%5D%2C%22type%22%3A%22group%22%7D%2C%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%22%3A%7B%22isOpen%22%3Atrue%2C%22knownContainerUniqueIds%22%3A%5B%22ngu_marinbunnsedimenter%22%5D%2C%22type%22%3A%22group%22%7D%2C%22%2F%22%3A%7B%22type%22%3A%22group%22%7D%2C%22%2F%2FMarint+Natur+Kart%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%7D%2C%22workbench%22%3A%5B%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%2FSedimentkornstorrelse%2FKornstorrelse_Det%22%2C%22%2F%2FMarint+Natur+Kart%2FBunntyper+Union+AOI+%22%5D%2C%22timeline%22%3A%5B%22ngu_marinbunnsedimenter%2FMarinBunnsedimenterWMS%2FSedimentkornstorrelse%2FKornstorrelse_Det%22%2C%22%2F%2FMarint+Natur+Kart%2FBunntyper+Union+AOI+%22%5D%2C%22initialCamera%22%3A%7B%22west%22%3A5.338282585144044%2C%22south%22%3A62.19159014075552%2C%22east%22%3A5.37830114364624%2C%22north%22%3A62.20154880517489%7D%2C%22homeCamera%22%3A%7B%22west%22%3A4%2C%22south%22%3A57.00000000000001%2C%22east%22%3A32%2C%22north%22%3A72%7D%2C%22viewerMode%22%3A%222d%22%2C%22showSplitter%22%3Afalse%2C%22splitPosition%22%3A0.5%2C%22settings%22%3A%7B%22baseMaximumScreenSpaceError%22%3A2%2C%22useNativeResolution%22%3Afalse%2C%22alwaysShowTimeline%22%3Afalse%2C%22baseMapId%22%3A%22basemap-openstreetmap%22%2C%22terrainSplitDirection%22%3A0%2C%22depthTestAgainstTerrainEnabled%22%3Afalse%7D%2C%22stories%22%3A%5B%5D%7D%5D%7D) there can be breaking changes for the link.

To make the code easy to run it reads data from a public accessible cloud storage bucket [gs://niva-geodata](https://console.cloud.google.com/storage/browser/niva-geodata/MarintNaturKart;tab=objects?project=nivaprod-1&prefix=&forceOnObjectsSortingFiltering=false) these dataset are work in progress.

In [1]:
import geopandas as gpd
import pandas as pd

import subkart

# Labelling

Use [bunnsedimenter](https://kartkatalog.geonorge.no/metadata/79f0f17d-9f62-456d-b1a9-2a8c754c51c4)dataset and NiN-LM from https://static.ngu.no/Mareano/Kornstorrelse.html to map into soft, hard, and mixed.

In [2]:
# https://static.ngu.no/Mareano/Kornstorrelse.html
# also see https://static.ngu.no/Mareano/Kornstorrelse-NiN-LM.html  DK_EFGY hard while, DK_ABCD soft and DK_0 mixture
HARD_BOTTOM_LM_DK = ["DK_EFGY"]
SOFT_BOTTOM_LM_DK = ["DK_AB", "DK_C", "DK_D"]
MIXED_BOTTOM_LM_DK = ["DK_0"]

In [3]:
df_kornstr = pd.read_html("https://static.ngu.no/Mareano/Kornstorrelse-NiN-LM.html", skiprows=0)[1]

In [4]:
SOFT_BOTTOM_TYPES = {
    row["Kornstørrelse SOSI-kode"]: row["Kornstørrelse SOSI-navn"]
    for _, row in df_kornstr[
        df_kornstr["Dominerende kornstørrelse LM-DK"].isin(SOFT_BOTTOM_LM_DK)
    ].iterrows()
}
HARD_BOTTOM_TYPES = {
    row["Kornstørrelse SOSI-kode"]: row["Kornstørrelse SOSI-navn"]
    for _, row in df_kornstr[
        df_kornstr["Dominerende kornstørrelse LM-DK"].isin(HARD_BOTTOM_LM_DK)
    ].iterrows()
}
MIXTURE_BOTTOM_TYPES = {
    row["Kornstørrelse SOSI-kode"]: row["Kornstørrelse SOSI-navn"]
    for _, row in df_kornstr[
        df_kornstr["Dominerende kornstørrelse LM-DK"].isin(MIXED_BOTTOM_LM_DK)
    ].iterrows()
}


df_types = pd.concat(
    {
        f"Løsbunn[ {', '.join(SOFT_BOTTOM_LM_DK)} ]": pd.DataFrame(
            list(SOFT_BOTTOM_TYPES.items()), columns=["SOSI-kode", "SOSI-navn"]
        ),
        f"Hardbunn[ {', '.join(HARD_BOTTOM_LM_DK)} ]": pd.DataFrame(
            list(HARD_BOTTOM_TYPES.items()), columns=["SOSI-kode", "SOSI-navn"]
        ),
        f"Blanding[ {', '.join(MIXED_BOTTOM_LM_DK)} ]": pd.DataFrame(
            list(MIXTURE_BOTTOM_TYPES.items()), columns=["SOSI-kode", "SOSI-navn"]
        ),
    },
    axis=1,
)

df_types.fillna('')

Løsbunn[ DK_AB, DK_C, DK_D ]                                  \
                      SOSI-kode                       SOSI-navn   
0                            10                            Leir   
1                            15                   Organisk slam   
2                            20                            Slam   
3                            21  Slam med blokker av sedimenter   
4                            30                 Sandholdig leir   
5                            40                 Sandholdig slam   
6                            50                            Silt   
7                            60                 Sandholdig silt   
8                            70                 Leirholdig sand   
9                            80                 Slamholdig sand   
10                           90                 Siltholdig sand   
11                           95                        Fin sand   
12                          100                            Sand   
13                          105                       Grov sand   
14                          110                 Grusholdig slam   
15                          115      Grusholdig sandholdig slam   
16                          120      Grusholdig slamholdig sand   
17                          130                 Grusholdig sand   
18                          140                 Slamholdig grus   
19                          150      Slamholdig sandholdig grus   
20                          160                 Sandholdig grus   
21                          170                            Grus   
22                          174                   Grus og stein   
23                          175            Grus, stein og blokk   

   Hardbunn[ DK_EFGY ]                                                     \
             SOSI-kode                                          SOSI-navn   
0                180.0                                     Stein og blokk   
1                300.0       Harde sedimenter eller sedimentære bergarter   
2                  1.0  Tynt eller usammenhengende sedimentdekke over ...   
3                  5.0                                         Bart fjell   
4                                                                           
5                                                                           
6                                                                           
7                                                                           
8                                                                           
9                                                                           
10                                                                          
11                                                                          
12                                                                          
13                                                                          
14                                                                          
15                                                                          
16                                                                          
17                                                                          
18                                                                          
19                                                                          
20                                                                          
21                                                                          
22                                                                          
23                                                                          

   Blanding[ DK_0 ]                                         
          SOSI-kode                              SOSI-navn  
0             185.0                    Sand, grus og stein  
1             190.0                          Sand og blokk  
2            

In [5]:
gdf_bunn = gpd.read_parquet(
    "gs://niva-geodata/MarintNaturKart/ngu_sediment/BunnsedimentKornstorDetalj.geo.parquet"
)

In [6]:
def to_bunn_type(kornstorrelse: int) -> str:
    if kornstorrelse in SOFT_BOTTOM_TYPES:
        return "løsbunn"
    elif kornstorrelse in HARD_BOTTOM_TYPES:
        return "fastbunn"
    elif kornstorrelse in MIXTURE_BOTTOM_TYPES:
        return "blanding"



In [7]:
gdf_bunn["BunnType"] = gdf_bunn["sedKornstørrelse"].map(to_bunn_type)

In [ ]:
fname = subkart.utils.to_filename("nisjedata-substrat-klassifisering", "norge", "latest", gdf_bunn.crs.to_epsg())

gdf_bunn.to_file(f"{fname}.geojson", driver="GeoJSON")
gdf_bunn.to_parquet(f"{fname}.geo.parquet", compression="snappy")

subkart.utils.to_postgis(gdf_bunn, fname)

## Clip to 2025 AOI

The coast line is created by fieldGeo. This creates a smaller dataset for the area of interest.

In [12]:
KOMMUNER = {
    "1515": "Herøy",
    "1516": "Ulstein",
    "1520": "Ørsta",
    "1577": "Volda",
    "1508": "Ålesund",
    "1532": "Giske",
    "1514": "Sande",
    "1531": "Sula",
    "1580": "Haram",
    "1528": "Sykkylven",
    "1517": "Hareid",
    "1511": "Vanylven",
}

gdf_kommuner = gpd.read_file(
    "https://storage.googleapis.com/niva-geodata/MarintNaturKart/kommuner_simplified.geojson"
).query("kommunenummer in @KOMMUNER")


In [13]:
sea_region = gpd.read_file("https://storage.googleapis.com/niva-geodata/MarintNaturKart/sea_union-from-HI-moere_og_romsdal.gpkg")

In [14]:
# Ensure matching CRS before clipping
if gdf_kommuner.crs != sea_region.crs:
    sea_region = sea_region.to_crs(gdf_kommuner.crs)

gdf_kommuner_sea = gpd.clip(gdf_kommuner, sea_region)

In [15]:
# Select polygons from gdf_bunn that are inside gdf_kommuner_sea
if gdf_bunn.crs != gdf_kommuner_sea.crs:
    gdf_bunn = gdf_bunn.to_crs(gdf_kommuner_sea.crs)

kommuner_sea_union = gdf_kommuner_sea.geometry.union_all()
gdf_mr = gdf_bunn[gdf_bunn.geometry.within(kommuner_sea_union)].copy()

print(f"Selected {len(gdf_mr)} of {len(gdf_bunn)} polygons inside kommuner sea mask.")

Selected 14029 of 130953 polygons inside kommuner sea mask.


## Create ploygons missing classification

Set all columns to 'missing' initally

In [16]:
bunn_union = gdf_mr.geometry.union_all()

missing_geom = kommuner_sea_union.difference(bunn_union)

gdf_missing_bunn_union = gpd.GeoDataFrame(geometry=[missing_geom], crs=gdf_kommuner_sea.crs)

missing_parts = gdf_missing_bunn_union.explode(index=False, ignore_index=True)

cols = [c for c in gdf_mr.columns if c != "geometry"]
missing_df = {c: ["missing"] * len(missing_parts) for c in cols}

gdf_missing_rows = gpd.GeoDataFrame(missing_df, geometry=missing_parts.geometry, crs=gdf_mr.crs)

# Append to existing selection
gdf_mr = gpd.GeoDataFrame(
    pd.concat([gdf_mr, gdf_missing_rows], ignore_index=True),
    geometry="geometry",
    crs=gdf_mr.crs,
)

In [17]:
# Create GeoDataFrame of missing areas with BunnType="missing" and other fields None,
# then append to gdf_bunn_in_kommuner_sea
missing_parts = gdf_missing_bunn_union.explode(index=False, ignore_index=True)

# Build a DataFrame with same columns as gdf_bunn_in_kommuner_sea
cols = [c for c in gdf_mr.columns if c != "geometry"]
missing_df = {c: ['missing'] * len(missing_parts) for c in cols}
missing_df["BunnType"] = ["missing"] * len(missing_parts)

gdf_missing_rows = gpd.GeoDataFrame(missing_df, geometry=missing_parts.geometry, crs=gdf_mr.crs)

gdf_mr = gpd.GeoDataFrame(
    pd.concat([gdf_mr, gdf_missing_rows], ignore_index=True),
    geometry="geometry",
    crs=gdf_mr.crs,
)


## Store inital labels

Store inital labels based on NGU data and missing area to reusable datasets. The geojson can be loaded into terriamap for comparison when exploring features, datasets and models more. 

In [18]:
fname = utils.to_filename("nisjedata-substrat-klassifisering", "moere-og-romsdal", "latest", gdf_mr.crs.to_epsg())
gdf_mr.to_file(f"{fname}.geojson", driver="GeoJSON")
gdf_mr.to_file(f"{fname}.gpkg", layer="soft_hard_bottom", driver="GPKG")
utils.to_postgis(gdf_mr, fname)

Table nisjedata-substrat-klassifisering_moere-og-romsdal uploaded to PostGIS.
